# ST-04 — Siretisation Phase 3 (matching approfondi)

Sur les EG non résolus en Phase 2 (DOUTEUX, REJETE, SANS_CANDIDAT au rang 1), matching approfondi avec :
- code APE (`cdape_stru` ↔ `activitePrincipaleUniteLegale`)
- année de création de l'établissement (`dtouverture_stru` ↔ `dateCreationEtablissement`)

Blocking élargi par **département**. Bonus +15 sur SIRET cohérent (avec exclusion EG jumeaux).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from tqdm.auto import tqdm

from src.siretisation import scorer_paire_approfondi_eg
from src.matching     import classifier_resultat
from src.excel_export import export_topn_excel, LABELS
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    ST_PERIMETRE, SIRENE_ETAB_FILTRE, ST_PHASE1, ST_PHASE2, ST_PHASE3, RESULTS_ST_DIR,
)
RESULTS_ST_DIR.mkdir(parents=True, exist_ok=True)

# Récupération des SIRET validés en P1 (pour désactivation bonus)
FEUILLES_VALIDES = {'Valide_fort', 'Valide'}
sheets_p1 = pd.read_excel(ST_PHASE1, sheet_name=None, dtype=str)
sirets_valides_p1 = set()
for nom, sdf in sheets_p1.items():
    if nom in FEUILLES_VALIDES and 'nmsiret_stru' in sdf.columns:
        sirets_valides_p1.update(
            sdf['nmsiret_stru'].dropna().astype(str)
            .str.replace(r'\s', '', regex=True).str.strip()
        )
sirets_valides_p1.discard('')
print(f'SIRET validés en P1 (pour désactivation bonus) : {len(sirets_valides_p1):,}')

## 1. Récupération des EG non résolus en Phase 2

In [2]:
NON_RES = {'DOUTEUX', 'REJETE', 'SANS_CANDIDAT'}

df_p2 = pd.read_excel(ST_PHASE2, sheet_name='Top5', dtype=str)
df_p2['rang'] = pd.to_numeric(df_p2['rang'], errors='coerce')
r1 = df_p2[df_p2['rang'] == 1]
ids_phase3 = set(r1[r1['statut_candidat'].isin(NON_RES)]['idstructure_stru']
                  .dropna().astype(str).tolist())

df_perimetre = pd.read_parquet(ST_PERIMETRE)
df_perimetre['idstructure_stru'] = df_perimetre['idstructure_stru'].astype(str)

# Rapatrier cdape_stru et dtouverture_stru depuis FINESS si manquants
if 'cdape_stru' not in df_perimetre.columns or 'dtouvertstruct_stru' not in df_perimetre.columns:
    print("Récupération de cdape_stru et dtouvertstruct_stru depuis FINESS...")
    from src.connexion import get_finess_connection
    conn = get_finess_connection()
    query = """
        SELECT idstructure_stru, cdape_stru, dtouvertstruct_stru
        FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
        WHERE topsource_stru = 'FINESS' AND typeidpm_stru = 'EG'
    """
    df_complement = pd.read_sql(query, conn)
    conn.close()
    df_complement['idstructure_stru'] = df_complement['idstructure_stru'].astype(str)
    df_perimetre = df_perimetre.merge(df_complement, on='idstructure_stru', how='left')

df_phase3 = df_perimetre[df_perimetre['idstructure_stru'].isin(ids_phase3)].copy().reset_index(drop=True)

df_etab = pd.read_parquet(SIRENE_ETAB_FILTRE)
df_etab['siret'] = df_etab['siret'].astype(str)

print(f'EG à traiter Phase 3 : {len(df_phase3):,}')
print(f'Etab SIRENE candidats: {len(df_etab):,}  (base filtrée)')

Récupération de cdape_stru et dtouvertstruct_stru depuis FINESS...


/tmp/ipykernel_352/1475320293.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_complement = pd.read_sql(query, conn)


EG à traiter Phase 3 : 13,162
Etab SIRENE candidats: 241,627  (base filtrée)


## 2. Indexation des Etab par département

In [3]:
etab_par_dept = df_etab.groupby('dept_etab', sort=False)
print(f'Départements avec au moins un Etab : {etab_par_dept.ngroups:,}')

Départements avec au moins un Etab : 105


## 3. Matching approfondi top 3

In [4]:
BONUS_SIRET_COHERENT = 15.0

lignes_top3 = []
lignes_orphelins = []

for _, row_eg in tqdm(df_phase3.iterrows(), total=len(df_phase3),
                       desc='Phase 3 matching approfondi'):
    dept = row_eg.get('dept_eg')
    siret_eg = str(row_eg.get('nmsiret_stru', '') or '').strip().replace(' ', '')
    bonus_applicable = (siret_eg != '') and (siret_eg not in sirets_valides_p1)

    if not dept or dept not in etab_par_dept.groups:
        lignes_orphelins.append({
            **row_eg.to_dict(),
            'siret_ref_app': None, 'siren_ref_app': None, 'nom_etab_retenu': None,
            'score_nom': None, 'score_adresse': None, 'score_global': None,
            'score_ape': None, 'score_date': None,
            'ape_eg': None, 'ape_etab': None,
            'annee_creation_eg': None, 'annee_creation_etab': None,
            'score_global_approfondi': None,
            'siret_coherent': False, 'bonus_applique': False,
            'score_approfondi_ajuste': None,
            'rang': 1, 'statut_candidat': 'SANS_CANDIDAT',
        })
        continue

    candidates = etab_par_dept.get_group(dept)

    scores = []
    for _, row_etab in candidates.iterrows():
        s = scorer_paire_approfondi_eg(row_eg, row_etab)
        siret_etab = str(row_etab['siret']).strip()
        coherent = bool(siret_eg) and (siret_eg == siret_etab)
        bonus = BONUS_SIRET_COHERENT if (coherent and bonus_applicable) else 0.0
        score_ajuste = min(s['score_global_approfondi'] + bonus, 100.0)

        scores.append({
            'siret_ref_app':                 row_etab['siret'],
            'siren_ref_app':                 row_etab.get('siren'),
            'denominationUniteLegale':       row_etab.get('denominationUniteLegale'),
            'enseigne1Etablissement':        row_etab.get('enseigne1Etablissement'),
            'enseigne2Etablissement':        row_etab.get('enseigne2Etablissement'),
            'enseigne3Etablissement':        row_etab.get('enseigne3Etablissement'),
            'denominationUsuelleEtablissement': row_etab.get('denominationUsuelleEtablissement'),
            'adresse_complete_etab':         row_etab.get('adresse_complete_etab'),
            'codeCommuneEtablissement':      row_etab.get('codeCommuneEtablissement'),
            'categorieJuridiqueUniteLegale': row_etab.get('categorieJuridiqueUniteLegale'),
            'activitePrincipaleUniteLegale': row_etab.get('activitePrincipaleUniteLegale'),
            'dateCreationEtablissement':     row_etab.get('dateCreationEtablissement'),
            **s,
            'siret_coherent':           coherent,
            'bonus_applique':           bool(bonus > 0),
            'score_approfondi_ajuste':  round(score_ajuste, 2),
        })

    scores.sort(key=lambda x: x['score_approfondi_ajuste'], reverse=True)
    for rang, sc in enumerate(scores[:3], start=1):
        statut = classifier_resultat(
            sc['score_approfondi_ajuste'], sc['score_nom'], sc['score_adresse']
        )
        lignes_top3.append({
            **row_eg.to_dict(), **sc,
            'rang': rang, 'statut_candidat': statut,
        })

df_top3 = pd.DataFrame(lignes_top3 + lignes_orphelins)

print(f'\nLignes top 3 Phase 3 : {len(df_top3):,}')
print(df_top3[df_top3['rang'] == 1]['statut_candidat'].value_counts())

Phase 3 matching approfondi:   0%|          | 0/13162 [00:00<?, ?it/s]


Lignes top 3 Phase 3 : 39,486
statut_candidat
DOUTEUX        10070
VALIDE          2110
REJETE           981
VALIDE_FORT        1
Name: count, dtype: int64


## 4. Aperçu

In [5]:
afficher_tableau(
    df_top3[['idstructure_stru', 'nmfinessej_stru', 'raisonsociale_stru',
             'siret_ref_app', 'denominationUniteLegale', 'nom_etab_retenu',
             'ape_eg', 'ape_etab', 'score_ape',
             'score_global', 'score_global_approfondi', 'score_approfondi_ajuste',
             'statut_candidat', 'rang']],
    'Aperçu top 3 Phase 3', max_lignes=9,
)

idstructure_stru,nmfinessej_stru,raisonsociale_stru,siret_ref_app,denominationUniteLegale,nom_etab_retenu,ape_eg,ape_etab,score_ape,score_global,score_global_approfondi,score_approfondi_ajuste,statut_candidat,rang
1931893,240011346,PHARMACIE DAUMARES,44236514400017,SYNDICAT COPROP SAINT MARTIN,SYNDICAT COPROP SAINT MARTIN,,8110Z,nan,57.300000,49.840000,49.840000,DOUTEUX,1
1931893,240011346,PHARMACIE DAUMARES,22240001200746,DEPARTEMENT DE LA DORDOGNE,CENTRE PLANIFICATION,,8411Z,nan,34.870000,47.900000,47.900000,DOUTEUX,2
1931893,240011346,PHARMACIE DAUMARES,22240001200407,DEPARTEMENT DE LA DORDOGNE,DISPENSAIRE POLYVALENT,,8411Z,nan,38.240000,46.590000,46.590000,DOUTEUX,3
1931898,240011411,PHARMACIE DEREINE,33099940000014,None,,,4773Z,nan,51.210000,60.970000,75.970000,VALIDE,1
1931898,240011411,PHARMACIE DEREINE,17240001200016,PREFECTURE DE DEPARTEMENT DORDOGNE,PREFECTURE DEPARTEMENT DORDOGNE,,8411Z,nan,64.040000,71.230000,71.230000,VALIDE,2
1931898,240011411,PHARMACIE DEREINE,22240001200019,DEPARTEMENT DE LA DORDOGNE,DEPARTEMENT DORDOGNE,,8411Z,nan,60.420000,68.340000,68.340000,VALIDE,3
1931913,240006403,SAMSAH TSA,77556982500281,ASSOC LES PAPILLONS BLANCS,ASSOC PAPILLONS BLANCS,9499Z,8720A,0.000000,50.370000,41.260000,56.260000,DOUTEUX,1
1931913,240006403,SAMSAH TSA,82778666600012,COMITE DE JUMELAGE BERGERAC-KENITRA,COMITE JUMELAGE BERGERAC KENITRA,9499Z,9499Z,100.000000,24.880000,47.420000,47.420000,DOUTEUX,2
1931913,240006403,SAMSAH TSA,92028140900010,CAP MAISON TRANSITION EN BERGERACOIS (POUR LE CLIMAT ET L'ENVIRONNEMENT),CAP MAISON TRANSITION BERGERACOIS CLIMAT ENVIRONNEMENT,9499Z,9499Z,100.000000,29.950000,46.970000,46.970000,DOUTEUX,3


## 5. Export Excel

In [6]:
COLS_EXPORT = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru',
    'categetab_stru', 'nmsiret_stru', 'raisonsociale_stru',
    'cdcommune_stru', 'adresse_complete_eg',
    'cdape_stru', 'dtouvertstruct_stru',
    'siret_ref_app', 'siren_ref_app', 'siret_coherent', 'bonus_applique',
    'denominationUniteLegale', 'enseigne1Etablissement', 'enseigne2Etablissement',
    'enseigne3Etablissement', 'denominationUsuelleEtablissement', 'nom_etab_retenu',
    'adresse_complete_etab', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale',
    'ape_eg', 'ape_etab', 'score_ape',
    'annee_creation_eg', 'annee_creation_etab', 'score_date',
    'score_nom', 'score_adresse', 'score_global',
    'score_global_approfondi', 'score_approfondi_ajuste',
    'statut_candidat', 'rang',
]

df_top3['rang'] = df_top3['rang'].astype(int)
compteurs = export_topn_excel(df_top3, ST_PHASE3, COLS_EXPORT, sheet_name='Top3')

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Phase 3 — matching approfondi (rang 1)')
print(f'\nFichier : {ST_PHASE3}')

Statut,Nb,% du total
Valide_fort,1,0.0%
Valide,"2,110",16.0%
Douteux,"10,070",76.5%
Rejeté,981,7.5%
Sans_candidat,0,0.0%
TOTAL,"13,162",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/siretisation/siretisation_phase3_approfondi.xlsx
